# **[Ablation Study 2] Evaluating Asymmetric Feature Matching Efficiency (yunnan-garden-pavilion-P71)**

### **Experiment Objective & Strategy**

##### **The Objective: Component Ablation Analysis**

With the core architecture validated in Phase 1, Phase 2 shifts focus to component-level optimization on unconstrained, real-world data. The objective of this ablation study is to quantify the computational trade-offs of the Symmetric Cross-Check heuristic. By bypassing this strict outlier filter using the `disable-symmetric-matching` flag, we aim to measure exactly how much compute time is saved, and whether the pipeline's geometric RANSAC algorithms can safely absorb the resulting influx of raw, asymmetric matches without breaking.

##### **The Field Evaluation Strategy & yunnan-garden-pavilion-P71**

To conduct this real-world evaluation, this ablation uses the **yunnan-garden-pavilion-P71** dataset. The dense, non-rigid foliage surrounding the pavilion acts as a massive generator for ambiguous, high-frequency micro-textures. Disabling the strict cross-check here is highly dangerous for the pipeline—it will intentionally flood the engine with false-positive matches from the shifting leaves. This serves as the ultimate stress test for the Perspective-n-Point (PnP) solver's ability to isolate rigid structural geometry from chaotic natural noise.

##### **Feature Matcher Evaluation (Ablated)**

During this ablation, the evaluation focuses on how fundamentally different matching architectures respond to the removal of the mutual nearest-neighbor constraint:
* **SIFT & ORB:** Classical feature matching relies on K-Nearest Neighbors (KNN). Bypassing the backward KNN cross-check inherently forces these matchers to accept a significantly higher volume of raw, asymmetric (and often false-positive) matches in exchange for reduced computational overhead. The critical metric for these architectures is the "Match Survival Rate"—a direct measurement of whether the downstream Perspective-n-Point (PnP) solver and RANSAC can efficiently reject this amplified noise, or if the false associations overwhelm the geometric tracking.
* **SuperPoint + LightGlue:** Unlike classical nearest-neighbor approaches, LightGlue utilizes Transformer-based cross-attention and optimal transport to compute match assignments. Because it evaluates correspondences using internal probability scores rather than external distance ratios or manual cross-checks, its architecture intrinsically ignores traditional symmetric filtering flags. Consequently, SuperPoint naturally functions as a stable control baseline for this experiment, remaining structurally unaffected by the ablation.

##### **Core Engine Configuration: `incremental-sfm`**

This notebook executes the same modular [`chuo-v/incremental-sfm`](https://github.com/chuo-v/incremental-sfm) engine utilized in the baseline studies. However, the pipeline is overridden at runtime using the newly introduced `--disable-symmetric-matching` command-line flag. This isolates the matching logic while keeping the underlying tracking and triangulation math identical for a clean comparison.

### **Infrastructure & Environment Setup**

In [1]:
import os
import sys

In [2]:
CORE_REPO_OWNER = "chuo-v"
CORE_REPO_NAME = "incremental-sfm"
CORE_TAG_NAME = "v1.2.0"

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(CORE_REPO_NAME):
        repo_url = f"https://github.com/{CORE_REPO_OWNER}/{CORE_REPO_NAME}.git"
        print(f"Cloning {CORE_REPO_NAME} ({CORE_TAG_NAME})...")
        !git clone -q -b {CORE_TAG_NAME} --depth 1 {repo_url} &> /dev/null

    os.chdir(CORE_REPO_NAME)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")

else:
    # Local Execution: Assume incremental-sfm is cloned as a sibling directory
    local_repo_path = os.path.abspath(f"../{CORE_REPO_NAME}")

    if os.path.exists(local_repo_path):
        os.chdir(local_repo_path)
        print(f"Environment Ready: Working directory set to {os.getcwd()}")
    else:
        print(f"Error: {CORE_REPO_NAME} not found at {local_repo_path}.")
        print("Please clone the core engine repository as a sibling to your current folder.")

Cloning incremental-sfm (v1.2.0)...
Environment Ready: Working directory set to /content/incremental-sfm


In [3]:
%%capture
!pip install -e .

### **Data Preparation & Camera Intrinsics**

In [4]:
from pathlib import Path

In [5]:
data_dir = Path("../datasets").resolve()
dataset_name = "yunnan-garden-pavilion-P71"

os.environ["DATA_DIR"] = str(data_dir)
os.environ["DATASET"] = dataset_name

with open(".env", "w") as f:
    f.write(f"DATA_DIR={data_dir}\n")

print(f"Configured DATA_DIR={data_dir} for dataset={dataset_name}")

Configured DATA_DIR=/content/datasets for dataset=yunnan-garden-pavilion-P71


In [6]:
DATASET_TARGET_DIR = data_dir / dataset_name
DATASET_REPO_OWNER = "chuo-v"
DATASET_REPO_NAME = "incremental-sfm-datasets"
DATASET_TAG_NAME = "v1.1.0"

if not DATASET_TARGET_DIR.exists():
    import shutil

    print(f"Downloading {dataset_name} dataset via sparse checkout ({DATASET_TAG_NAME})...")
    repo_url = f"https://github.com/{DATASET_REPO_OWNER}/{DATASET_REPO_NAME}.git"

    !git clone -q -b {DATASET_TAG_NAME} --depth 1 --filter=blob:none --sparse {repo_url} temp_datasets_repo &> /dev/null
    !cd temp_datasets_repo && git sparse-checkout set {dataset_name} &> /dev/null

    DATASET_TARGET_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(f"temp_datasets_repo/{dataset_name}", str(DATASET_TARGET_DIR))
    shutil.rmtree("temp_datasets_repo")

    print(f"Dataset ready at: {DATASET_TARGET_DIR}")
else:
    print(f"Dataset {dataset_name} already exists at {DATASET_TARGET_DIR}.")

Dataset ready at: /content/datasets/yunnan-garden-pavilion-P71


### **Pipeline Execution**

##### **SIFT Execution**

Executes the pipeline using SIFT (Scale-Invariant Feature Transform) to establish a baseline for the limitations of traditional computer vision in unconstrained environments. For this dataset, SIFT is specifically evaluated on its vulnerability to dense background foliage, measuring how high-frequency natural clutter and non-rigid elements (like shifting leaves) cause aggressive keypoint culling during Lowe's ratio test and trigger frequent sequential tracking failures.

In [7]:
!python main.py \
    --dataset {dataset_name} \
    --matcher SIFT \
    --downscale-factor 0.5 \
    --max-cache-size 100 \
    --max-features 50000 \
    --lowe-ratio-threshold 0.80 \
    --apply-clahe \
    --ransac-prob 0.999 \
    --ransac-threshold 6.0 \
    --min-parallax-degrees 1.0 \
    --disable-symmetric-matching

2026-10-14 00:10:33,229 - INFO - ==============================
2026-10-14 00:10:33,229 - INFO - Starting Incremental SfM Pipeline
2026-10-14 00:10:33,229 - INFO - Dataset: yunnan-garden-pavilion-P71
2026-10-14 00:10:33,229 - INFO - Matcher: SIFT
2026-10-14 00:10:33,229 - INFO - Data Directory: /content/datasets
2026-10-14 00:10:33,229 - INFO - ==============================
2026-10-14 00:10:33,247 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-14 00:11:50,607 - INFO - Two-View Initialization complete (77.36s). Seeded 1463 3D points.
2026-10-14 00:11:50,607 - INFO - Processing Image 2 / 70...
2026-10-14 00:13:04,889 - INFO - Registered Image 2 (74.282s): 621 PnP Inliers | 920 Triangulated Points
2026-10-14 00:13:04,889 - INFO - Processing Image 3 / 70...
2026-10-14 00:14:02,961 - INFO - Registered Image 3 (58.071s): 497 PnP Inliers | 817 Triangulated Points
2026-10-14 00:14:02,961 - INFO - Processing Image 4 / 70...
2026-10-14 00:14:49,828 - INFO - Registered Image 4 (

##### **SuperPoint Execution**

Integrates PyTorch-driven SuperPoint and LightGlue to evaluate the resilience of modern deep learning architectures against real-world scene ambiguity. This execution tests the ability of semantic awareness and Transformer-based cross-attention to successfully deprioritize noisy, natural textures in the background and focus keypoint extraction purely on the rigid architecture of the pavilion, maintaining sequential tracking where traditional hand-crafted descriptors starve.

In [8]:
!python main.py \
    --dataset {dataset_name} \
    --matcher SuperPoint \
    --downscale-factor 0.5 \
    --max-cache-size 100 \
    --max-features 50000 \
    --lowe-ratio-threshold 0.80 \
    --apply-clahe \
    --ransac-prob 0.999 \
    --ransac-threshold 6.0 \
    --min-parallax-degrees 1.0 \
    --disable-symmetric-matching

2026-10-14 01:20:49,910 - INFO - ==============================
2026-10-14 01:20:49,910 - INFO - Starting Incremental SfM Pipeline
2026-10-14 01:20:49,910 - INFO - Dataset: yunnan-garden-pavilion-P71
2026-10-14 01:20:49,910 - INFO - Matcher: SuperPoint
2026-10-14 01:20:49,910 - INFO - Data Directory: /content/datasets
2026-10-14 01:20:49,910 - INFO - ==============================
2026-10-14 01:20:49,912 - INFO - Initializing SuperPoint & LightGlue on device: cpu
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth" to /root/.cache/torch/hub/checkpoints/superpoint_v1.pth
100% 4.96M/4.96M [00:00<00:00, 22.1MB/s]
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_lightglue.pth" to /root/.cache/torch/hub/checkpoints/superpoint_lightglue_v0-1_arxiv.pth
100% 45.3M/45.3M [00:01<00:00, 44.7MB/s]
2026-10-14 01:20:52,463 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-14 01:21:02,183 - INFO - Two-View I

##### **ORB Execution**

Utilizes the lightweight ORB binary descriptor to evaluate the pipeline's high-speed computational efficiency on challenging field data. This execution measures whether fast corner detection combined with aggressive downscaling can effectively act as a low-pass filter—smoothing out the high-frequency environmental noise from the surrounding greenery to extract stable structural geometry from the pavilion.

In [9]:
!python main.py \
    --dataset {dataset_name} \
    --matcher ORB \
    --downscale-factor 0.5 \
    --max-cache-size 100 \
    --max-features 50000 \
    --lowe-ratio-threshold 0.80 \
    --apply-clahe \
    --ransac-prob 0.999 \
    --ransac-threshold 6.0 \
    --min-parallax-degrees 1.0 \
    --disable-symmetric-matching

2026-10-14 01:29:15,854 - INFO - ==============================
2026-10-14 01:29:15,854 - INFO - Starting Incremental SfM Pipeline
2026-10-14 01:29:15,854 - INFO - Dataset: yunnan-garden-pavilion-P71
2026-10-14 01:29:15,854 - INFO - Matcher: ORB
2026-10-14 01:29:15,854 - INFO - Data Directory: /content/datasets
2026-10-14 01:29:15,854 - INFO - ==============================
2026-10-14 01:29:15,856 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-14 01:29:21,850 - INFO - Two-View Initialization complete (5.994s). Seeded 577 3D points.
2026-10-14 01:29:21,850 - INFO - Processing Image 2 / 70...
2026-10-14 01:29:27,418 - INFO - Registered Image 2 (5.568s): 105 PnP Inliers | 349 Triangulated Points
2026-10-14 01:29:27,419 - INFO - Processing Image 3 / 70...
2026-10-14 01:29:33,107 - INFO - Registered Image 3 (5.688s): 75 PnP Inliers | 245 Triangulated Points
2026-10-14 01:29:33,107 - INFO - Processing Image 4 / 70...
2026-10-14 01:29:38,664 - INFO - Registered Image 4 (5.557

### **Quantitative Metrics**

This section aggregates and visualizes the JSON output metrics generated by each feature matcher. By comparing point cloud density, Mean Reprojection Error (RMSE), and average processing time side-by-side, we can quantitatively evaluate the engineering trade-offs between classical robustness (SIFT), AI-driven resilience (SuperPoint), and computational efficiency (ORB).

Additionally, this section computes and stores four high-level multi-view structural metrics:
* **Mean Track Length:** Measures geometric stability by averaging how many camera views observe each 3D point.
* **Tracking Inlier Ratio:** Measures descriptor discriminative power by evaluating the percentage of putative matches that geometrically survive Perspective-n-Point (PnP) tracking.
* **RMSE Variance:** Measures structural tension by analyzing the frame-by-frame variance of reprojection errors, identifying tracking spikes hidden by the overall mean.
* **Global Relocalizations:** Measures tracking robustness by counting how many times the pipeline lost tracking and had to perform a costly global relocalization to recover the camera pose.

In [10]:
%%capture
!pip install kaleido==0.2.1 plotly==5.24.1

In [11]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [12]:
matchers = ["SIFT", "SuperPoint", "ORB"]
metrics_data = []

for matcher in matchers:
    metrics_path = data_dir / dataset_name / "output" / "metrics" / f"run_metrics_{matcher}.json"

    if metrics_path.exists():
        with open(metrics_path, "r+") as f:
            data = json.load(f)
            summary = data["results"]["summary"]
            frames = data["results"]["frames"]

            total_3d_points = summary.get("total_3d_points", 0)
            sum_raw_matches = sum(frame.get("raw_matches", 0) for frame in frames)
            sum_pnp_inliers = sum(frame.get("pnp_inliers", 0) for frame in frames)
            sum_associations = sum(frame.get("associations", 0) for frame in frames)
            frame_rmses = [frame.get("rmse", 0.0) for frame in frames if "rmse" in frame]
            match_survival_rate = (sum_pnp_inliers / sum_raw_matches) if sum_raw_matches > 0 else 0.0

            # Every new point starts with 2 observations. PnP inliers add 1 observation to an existing point.
            mean_track_length = 2.0 + (sum_pnp_inliers / total_3d_points) if total_3d_points > 0 else 0.0
            tracking_inlier_ratio = (sum_pnp_inliers / sum_associations) if sum_associations > 0 else 0.0
            rmse_variance = float(np.var(frame_rmses)) if frame_rmses else 0.0
            global_relocs = sum(1 for frame in frames if frame.get("is_relocalized", False))

            summary["total_raw_matches"] = sum_raw_matches
            summary["match_survival_rate"] = match_survival_rate
            summary["tracking_inlier_ratio"] = tracking_inlier_ratio
            summary["mean_track_length"] = mean_track_length
            summary["rmse_variance"] = rmse_variance
            summary["global_relocalizations"] = global_relocs

            f.seek(0)
            json.dump(data, f, indent=4)
            f.truncate()

            metrics_data.append({
                "Matcher": matcher,
                "Total 3D Points": total_3d_points,
                "Total Raw Matches": sum_raw_matches,
                "Overall RMSE (px)": round(summary.get("overall_rmse", 0.0), 3),
                "RMSE Variance": round(rmse_variance, 3),
                "Avg Frame Time (s)": round(summary.get("avg_frame_time_seconds", 0.0), 3),
                "Mean Track Length": round(mean_track_length, 3),
                "Tracking Inlier Ratio (%)": round(tracking_inlier_ratio * 100, 1),
                "Match Survival Rate (%)": round(match_survival_rate * 100, 1),
                "Global Relocalizations": global_relocs
            })
    else:
        print(f"Metrics not found for {matcher} at {metrics_path}")

In [13]:
df = pd.DataFrame(metrics_data)
print(f"Pipeline Summary ({dataset_name}):\n")
print(df.to_string(index=False))

Pipeline Summary (yunnan-garden-pavilion-P71):

   Matcher  Total 3D Points  Total Raw Matches  Overall RMSE (px)  RMSE Variance  Avg Frame Time (s)  Mean Track Length  Tracking Inlier Ratio (%)  Match Survival Rate (%)  Global Relocalizations
      SIFT            97594             237524              4.013          3.487              59.936              2.400                       70.9                     16.4                       0
SuperPoint            69112             157441              3.703          0.070               7.103              2.565                       55.4                     24.8                       0
       ORB            51108             187250              3.332          0.678               7.091              2.221                       72.0                      6.0                       2


In [14]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        "Total 3D Points",
        "Overall RMSE",
        "Avg Frame Time",
        "Mean Track Length",
        "Tracking Inlier Ratio %",
        "RMSE Variance"
    ),
    vertical_spacing=0.15
)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, row in df.iterrows():
    m_color = colors[i]
    m_name = row["Matcher"]

    # Row 1
    fig.add_trace(go.Bar(x=[m_name], y=[row["Total 3D Points"]], marker_color=m_color, name=m_name, showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=[m_name], y=[row["Overall RMSE (px)"]], marker_color=m_color, showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=[m_name], y=[row["Avg Frame Time (s)"]], marker_color=m_color, showlegend=False), row=1, col=3)

    # Row 2
    fig.add_trace(go.Bar(x=[m_name], y=[row["Mean Track Length"]], marker_color=m_color, showlegend=False), row=2, col=1)
    fig.add_trace(go.Bar(x=[m_name], y=[row["Tracking Inlier Ratio (%)"]], marker_color=m_color, showlegend=False), row=2, col=2)
    fig.add_trace(go.Bar(x=[m_name], y=[row["RMSE Variance"]], marker_color=m_color, showlegend=False), row=2, col=3)

fig.update_layout(
    title_text=f"Quantitative Comparison: {dataset_name}",
    width=1200,
    height=750,
    template="plotly_white"
)
fig.show()

# Export figure
vis_dir = data_dir / dataset_name / "output" / "visualizations"
vis_dir.mkdir(parents=True, exist_ok=True)
bar_chart_path = vis_dir / f"metrics_bar_chart_{dataset_name}.png"
fig.write_image(str(bar_chart_path), scale=2)
print(f"\nBar chart exported to: {bar_chart_path}")


Bar chart exported to: /content/datasets/yunnan-garden-pavilion-P71/output/visualizations/metrics_bar_chart_yunnan-garden-pavilion-P71.png


In [15]:
max_pts = df["Total 3D Points"].max() if not df.empty else 1

# Calculate dynamic padding for the y-axis
y_min = df["Overall RMSE (px)"].min() if not df.empty else 0
y_max = df["Overall RMSE (px)"].max() if not df.empty else 1
y_pad = (y_max - y_min) * 0.25 if y_max > y_min else 0.5

# Calculate dynamic padding for the x-axis
x_min = df["Avg Frame Time (s)"].min() if not df.empty else 0
x_max = df["Avg Frame Time (s)"].max() if not df.empty else 1
x_pad = (x_max - x_min) * 0.25 if x_max > x_min else 0.5

fig_tradeoff = go.Figure()

for i, row in df.iterrows():
    matcher = row["Matcher"]

    fig_tradeoff.add_trace(go.Scatter(
        x=[row["Avg Frame Time (s)"]],
        y=[row["Overall RMSE (px)"]],
        mode='markers+text',
        name=matcher,
        text=[f"<b>{matcher}</b><br>{row['Total 3D Points']} pts<br>Track Length: {row['Mean Track Length']}<br>Inliers: {row['Tracking Inlier Ratio (%)']}%"],
        textposition="top center",
        cliponaxis=False,
        marker=dict(
            size=[row["Total 3D Points"]],
            sizemode='area',
            sizeref=2.0 * max_pts / (50.0 ** 2) if max_pts > 0 else 1,
            sizemin=10,
            color=colors[i],
            line=dict(width=2, color='DarkSlateGrey')
        )
    ))

fig_tradeoff.update_layout(
    title=f"Algorithmic Trade-Offs: Compute Time vs. Accuracy ({dataset_name})",
    xaxis_title="Average Frame Time (s)",
    yaxis_title="Overall RMSE (px)",
    xaxis=dict(range=[x_min - x_pad, x_max + x_pad]),
    yaxis=dict(range=[y_min - (y_pad * 0.5), y_max + y_pad]),
    template="plotly_white",
    width=800,
    height=500,
    showlegend=False,
    margin=dict(t=80, b=60, l=60, r=60)
)

fig_tradeoff.show()

# Export figure
tradeoff_chart_path = vis_dir / f"metrics_tradeoff_chart_{dataset_name}.png"
fig_tradeoff.write_image(str(tradeoff_chart_path), scale=2)
print(f"\nTrade-off chart exported to: {tradeoff_chart_path}")


Trade-off chart exported to: /content/datasets/yunnan-garden-pavilion-P71/output/visualizations/metrics_tradeoff_chart_yunnan-garden-pavilion-P71.png


### **3D Visualization & Filtering**

This section loads the raw 3D point clouds and applies statistical and radius-based outlier removal via Open3D to eliminate noise and spurious geometry. The cleaned point clouds are then rendered as full-width, interactive 3D plots using Plotly, allowing for a direct qualitative assessment of each feature matcher's structural accuracy and visual completeness.

In [16]:
%%capture
!pip install open3d

In [17]:
import open3d as o3d

In [18]:
def visualize_point_cloud(
    dataset_name: str,
    matcher: str,
    nb_neighbors: int = 20,
    std_ratio: float = 2.0,
    nb_points: int = 3,
    radius: float = 1.0
):
    """Loads, filters, renders, and exports a full-width interactive 3D point cloud."""
    pc_dir = data_dir / dataset_name / "output" / "point_clouds"
    vis_dir = data_dir / dataset_name / "output" / "visualizations"
    vis_dir.mkdir(parents=True, exist_ok=True)

    ply_path = pc_dir / f"point_cloud_{matcher}.ply"

    if not ply_path.exists():
        print(f"[{matcher}] PLY file not found: {ply_path}")
        return

    print(f"[{matcher}] Loading point cloud from {ply_path}...")
    pcd = o3d.io.read_point_cloud(str(ply_path))

    if pcd.is_empty():
        print(f"[{matcher}] Point cloud is empty.")
        return

    original_count = len(pcd.points)

    # Apply statistical and radius outlier removal
    print(f"[{matcher}] Running outlier removal...")
    pcd_filtered, _ = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    pcd_filtered, _ = pcd_filtered.remove_radius_outlier(nb_points=nb_points, radius=radius)

    filtered_count = len(pcd_filtered.points)
    print(f"[{matcher}] Filtered {original_count - filtered_count} extreme outliers. Remaining: {filtered_count} pts.")

    # Extract coordinates and standard 8-bit RGB colors
    points = np.asarray(pcd_filtered.points)
    colors_normalized = np.asarray(pcd_filtered.colors)
    colors_rgb = (colors_normalized * 255).astype(int)
    colors_str = [f'rgb({r}, {g}, {b})' for r, g, b in colors_rgb]

    # Render full-width interactive 3D plot
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers',
        marker=dict(size=2, color=colors_str, opacity=0.8),
        name=matcher
    )])
    fig.update_layout(
        title=f"{matcher} Reconstruction ({dataset_name}) - {filtered_count} pts",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(up=dict(x=0, y=0, z=1), center=dict(x=0, y=0, z=0), eye=dict(x=0, y=-1.5, z=1.5))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=800
    )

    # Export figure
    html_path = vis_dir / f"point_cloud_{matcher}_visualization.html"
    fig.write_html(str(html_path))
    print(f"[{matcher}] Interactive 3D plot exported to: {html_path}")

    # Open the interactive plot
    fig.show()

##### **SIFT**

In [19]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="SIFT",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=20,
    radius=0.5
)

[SIFT] Loading point cloud from /content/datasets/yunnan-garden-pavilion-P71/output/point_clouds/point_cloud_SIFT.ply...
[SIFT] Running outlier removal...
[SIFT] Filtered 26950 extreme outliers. Remaining: 70644 pts.
[SIFT] Interactive 3D plot exported to: /content/datasets/yunnan-garden-pavilion-P71/output/visualizations/point_cloud_SIFT_visualization.html


##### **SuperPoint**

In [20]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="SuperPoint",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=20,
    radius=0.85
)

[SuperPoint] Loading point cloud from /content/datasets/yunnan-garden-pavilion-P71/output/point_clouds/point_cloud_SuperPoint.ply...
[SuperPoint] Running outlier removal...
[SuperPoint] Filtered 7424 extreme outliers. Remaining: 61688 pts.
[SuperPoint] Interactive 3D plot exported to: /content/datasets/yunnan-garden-pavilion-P71/output/visualizations/point_cloud_SuperPoint_visualization.html


##### **ORB**

In [21]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="ORB",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=100,
    radius=1.0
)

[ORB] Loading point cloud from /content/datasets/yunnan-garden-pavilion-P71/output/point_clouds/point_cloud_ORB.ply...
[ORB] Running outlier removal...
[ORB] Filtered 13835 extreme outliers. Remaining: 37273 pts.
[ORB] Interactive 3D plot exported to: /content/datasets/yunnan-garden-pavilion-P71/output/visualizations/point_cloud_ORB_visualization.html
